In [1]:
import sys
from pathlib import Path
NOTEBOOKS = Path.cwd()                          # <-- Edit as needed
NN_POSTPROCESSING = NOTEBOOKS.parents[0]        # <-- Edit as needed
BAMBOO_SETUP = NOTEBOOKS.parents[3]             # <-- Edit as needed
Z_OUTPUT_eos = Path('/Users/antonett/Documents/HH Analysis/Z_OUTPUT_eos')   # <-- Edit as needed
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
%load_ext autoreload

In [2]:
from post_processing.NN.DataHandler import DataHandler
import logging
datahandler = DataHandler(
    workdir=Z_OUTPUT_eos/'2022_even_1013/Reco',
    tree_name='SL_res_2b_x',
    total_inputs= NN_POSTPROCESSING / 'input/vars40_new.txt',
    log_level=logging.DEBUG
)
# total_df = datahandler.start()
total_df_unprep = datahandler.load_data()
total_df_unprep = datahandler.fix_column_names_mismatch(total_df_unprep)
datahandler.data_inspection(total_df_unprep)
# datahandler.data_summary(total_df_unprep)
df = datahandler.preprocess_data(total_df_unprep)
datahandler.data_inspection(df)


In module 'Darwin':
/Library/Developer/CommandLineTools/SDKs/MacOSX15.0.sdk/usr/include/libkern/arm/OSByteOrder.h:14:1: error: '_OSSwapInt16' has different definitions in different modules; definition in module 'Darwin.libkern.OSByteOrder' first difference is return type is 'uint16_t' (aka 'unsigned short')
uint16_t
^~~~~~~~
/Library/Developer/CommandLineTools/SDKs/MacOSX15.0.sdk/usr/include/libkern/arm/_OSByteOrder.h:49:1: note: but in 'DarwinFoundation.OSByteOrder' found different return type '__uint16_t' (aka 'unsigned short')
__uint16_t
^~~~~~~~~~
In module 'Darwin':
/Library/Developer/CommandLineTools/SDKs/MacOSX15.0.sdk/usr/include/libkern/arm/OSByteOrder.h:24:1: error: '_OSSwapInt32' has different definitions in different modules; definition in module 'Darwin.libkern.OSByteOrder' first difference is return type is 'uint32_t' (aka 'unsigned int')
uint32_t
^~~~~~~~
/Library/Developer/CommandLineTools/SDKs/MacOSX15.0.sdk/usr/include/libkern/arm/_OSByteOrder.h:59:1: note: but in 'Da

# Set up your model config

In [18]:
from post_processing.NN.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=True,
    residual_network=True,
    hiddenlayers=[
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    ],
    outputlayers=[
        {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 35, "validation_split": 0.25}
)

In [32]:
%autoreload 2

# Run DNN

In [34]:
from post_processing.NN.DNNModel import DNNModel, draw_all_stats
import post_processing.NN.utils as utils

DNN = DNNModel(model_config=model_config, modeldir= NOTEBOOKS/'model_custom', log_level=logging.DEBUG)
model_df = DNN.set_model_df_from_total_df(df)
X_train, X_test, Y_train, Y_test, evs_test, sw_train = DNN.Full_Splitting(model_df)
input_layer, normalized_input = DNN.input_preprocessing(X_train)

DNN.build_model(input_layer=input_layer, normalized_input=normalized_input, fixed_random_seed=True)
DNN.train_model(X_train, Y_train, sw_train)
DNN.save_model_info(X_train.columns, Y_train, Y_test)
output_df, model_metrics = DNN.evaluate_and_predict(X_test, Y_test, evs_test)
cm_norm_true, cm_norm_pred, diag_names = draw_all_stats(DNN_type=DNN.type, history=DNN.history, output_df=output_df, modeldir=DNN.modeldir, classes=DNN.classes)

Initializing model: multi_HH_ttbar_tW
HH_bbWW
Total sum of genWeights for HH_bbWW: 137.84710693359375 
Total sum of sample_weights for HH_bbWW: 2340784.0 
ttbar
Total sum of genWeights for ttbar: 414462528.0 
Total sum of sample_weights for ttbar: 18726300.0 
tW
Total sum of genWeights for tW: 10632820.0 
Total sum of sample_weights for tW: 9363136.0 

Preprocessing input ...

Building model ...
Sumary of model compiled


Model: "multi_HH_ttbar_tW"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 40)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 40)        │          0 │ input[0][0]       │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_0 (Dense)     │ (None, 64)        │      2,624 │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ layer_0[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_1 (Dense)     │ (None, 64)        │      4,160 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ layer_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64)        │          0 │ dropout_13[0][0], │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_2 (Dense)     │ (None, 64)        │      4,160 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ layer_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 3)         │        195 │ dropout_14[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,907 (46.51 KB)

 Trainable params: 11,523 (45.01 KB)

 Non-trainable params: 384 (1.50 KB)

None

Training model ...


Epoch 1/35
 829/1372 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6984 - auc_pr: 0.6949 - auc_roc: 0.8282 - f1_score_macro: 0.3413 - f1_score_micro: 0.6984 - loss: 16.8983 - precision: 0.7654 - precision_HH: 0.0067 - precision_tW: 0.3301 - precision_ttbar: 0.8665 - recall: 0.4770 - recall_HH: 0.0471 - recall_tW: 0.0799 - recall_ttbar: 0.5481

In [28]:
Y_train

,Class_HH,Class_tW,Class_ttbar
1109633,False,False,True
1016620,False,False,True
945513,False,False,True
712030,False,False,True
1188746,False,False,True
...,...,...,...
1087374,False,False,True
476944,False,False,True
1019360,False,False,True
176099,False,True,False


In [30]:
DNN.classes

['HH', 'ttbar', 'tW']

In [31]:
def run_custom(DNNObject, compiled_model):
    DNNObject.model = compiled_model
    DNNObject.train_model(X_train, Y_train, sw_train)
    output_df, model_metrics = DNNObject.evaluate_and_predict(X_test, Y_test, evs_test)
    utils.draw_score_distribution(DNNObject.type, output_df, DNNObject.modeldir)

,Class_HH,Class_tW,Class_ttbar
836358,False,False,True
1357097,False,False,True
1313274,False,False,True
1954390,False,False,True
735623,False,False,True
...,...,...,...
128457,False,True,False
2270162,False,False,True
850426,False,False,True
414996,False,False,True
